# 11. Results Summary — 전체 파이프라인 포트폴리오

전체 ML 파이프라인(04~10)의 결과를 한눈에 보여주는 요약 노트북입니다.

| Section | Content | Visualization |
|---------|---------|---------------|
| 1 | 결과 JSON 로드 | - |
| 2 | Master Table | go.Table |
| 3 | Progression Curve | Bar + Line |
| 4 | Waterfall Chart | Waterfall |
| 5 | Quality vs Speed | Bubble Scatter |
| 6 | Per-Domain Heatmap | Heatmap |
| 7 | Radar Chart | Radar |
| 8 | GGUF Pareto | Scatter + Pareto |
| 9 | Summary | Text |

---
## Executive Summary

> **프로젝트 한 줄 요약**: 35만 건 민원 데이터로 **분류 → 검색 → 답변 생성 → 배포**까지 end-to-end ML 파이프라인을 구축하고, 각 단계에서 baseline 대비 개선을 실험적으로 검증했습니다.

### 핵심 성과

| # | 성과 | 상세 |
|---|------|------|
| 1 | **분류 성능 향상** | TF-IDF + SVC baseline에서 KoELECTRA fine-tuning으로 Domain Macro F1 개선 |
| 2 | **검색 품질 향상** | BM25 키워드 검색에서 SBERT 의미 검색으로 Recall@5 대폭 개선 |
| 3 | **답변 품질 향상** | Zero-shot RAG에서 QLoRA fine-tuned 모델로 BERTScore 개선, GGUF 양자화로 경량 배포 |

### 비즈니스 임팩트

- **응답 자동화**: 민원 유형 자동 분류 + 유사 사례 검색 + 답변 초안 생성으로 상담원 업무 부하 감소
- **일관성**: 동일 유형 민원에 대해 일관된 품질의 답변 제공
- **확장성**: GGUF 양자화 모델로 GPU 없이 CPU 서빙 가능 → 인프라 비용 절감

---
## 0. Environment Setup

In [ ]:
%%capture
!pip install -q plotly kaleido

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'
warnings.filterwarnings('ignore')

# --- Kaggle vs Local path detection ---
if os.path.exists('/kaggle/input'):
    OUT_DIR = '/kaggle/working'
    IS_KAGGLE = True
else:
    OUT_DIR = '..'
    IS_KAGGLE = False

RESULTS_DIR = os.path.join(OUT_DIR, 'results')
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Results: {RESULTS_DIR}")

---
## 1. 결과 파일 로드

각 노트북에서 저장한 JSON 파일을 로드합니다. 파일이 없으면 graceful skip합니다.

In [ ]:
# --- Load all result files with graceful fallback ---
RESULT_FILES = {
    'ml_classification': 'classification_ml_results.json',
    'dl_classification': 'classification_dl_results.json',
    'bm25_retrieval': 'retrieval_bm25_results.json',
    'sbert_retrieval': 'retrieval_sbert_results.json',
    'rag_generation': 'generation_rag_results.json',
    'lora_generation': 'generation_lora_results.json',
    'gguf_deploy': 'generation_gguf_results.json',
}

results = {}
for key, filename in RESULT_FILES.items():
    path = os.path.join(RESULTS_DIR, filename)
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            results[key] = json.load(f)
        print(f"  [OK] {filename}")
    else:
        results[key] = None
        print(f"  [SKIP] {filename} — not found")

loaded = sum(1 for v in results.values() if v is not None)
print(f"\nLoaded: {loaded}/{len(RESULT_FILES)} result files")

---
## 2. Master Table — 전 단계 핵심 지표

In [ ]:
# --- Build Master Table ---
master_rows = []

# Phase 1: Classification
if results['ml_classification']:
    best_ml = results['ml_classification']['best_models']['domain']
    master_rows.append({
        'Phase': 'Classification',
        'Stage': '04. ML Baseline',
        'Model': best_ml['model'],
        'Primary Metric': 'Macro F1',
        'Score': best_ml['macro_f1'],
        'Secondary': f"Acc={best_ml.get('accuracy', 'N/A')}",
    })

if results['dl_classification']:
    best_dl = results['dl_classification']['best_models']['domain']
    master_rows.append({
        'Phase': 'Classification',
        'Stage': '05. Deep (KoELECTRA)',
        'Model': best_dl['model'],
        'Primary Metric': 'Macro F1',
        'Score': best_dl['macro_f1'],
        'Secondary': f"Acc={best_dl.get('accuracy', 'N/A')}",
    })

# Phase 2: Retrieval
if results['bm25_retrieval']:
    bm25 = results['bm25_retrieval']
    r5 = bm25.get('recall_at_5', bm25.get('metrics', {}).get('recall_at_5', 0))
    mrr = bm25.get('mrr', bm25.get('metrics', {}).get('mrr_at_10', 0))
    master_rows.append({
        'Phase': 'Retrieval',
        'Stage': '06. BM25',
        'Model': 'BM25Okapi',
        'Primary Metric': 'Recall@5',
        'Score': r5,
        'Secondary': f"MRR@10={mrr}",
    })

if results['sbert_retrieval']:
    sbert = results['sbert_retrieval']
    master_rows.append({
        'Phase': 'Retrieval',
        'Stage': '07. SBERT+FAISS',
        'Model': sbert.get('best_model', 'ko-sbert-nli'),
        'Primary Metric': 'Recall@5',
        'Score': sbert.get('recall_at_5', 0),
        'Secondary': f"MRR@5={sbert.get('mrr', 0)}",
    })

# Phase 3: Generation
if results['rag_generation']:
    rag = results['rag_generation']
    master_rows.append({
        'Phase': 'Generation',
        'Stage': '08. RAG Pipeline',
        'Model': rag.get('llm_model', 'N/A'),
        'Primary Metric': 'BERTScore',
        'Score': rag['metrics'].get('rag_bertscore', 0),
        'Secondary': f"ROUGE-L={rag['metrics'].get('rag_rouge_l', 0)}",
    })

if results['lora_generation']:
    lora = results['lora_generation']
    master_rows.append({
        'Phase': 'Generation',
        'Stage': '09. QLoRA',
        'Model': lora.get('base_model', 'Qwen2.5-7B'),
        'Primary Metric': 'BERTScore',
        'Score': lora['metrics'].get('lora_bertscore', 0),
        'Secondary': f"ROUGE-L={lora['metrics'].get('lora_rouge_l', 0)}",
    })

# Phase 4: Deployment
if results['gguf_deploy']:
    gguf = results['gguf_deploy']
    best_qt = gguf.get('best_quantization', 'N/A')
    qt_info = gguf.get('quantizations', {}).get(best_qt, {})
    master_rows.append({
        'Phase': 'Deployment',
        'Stage': f'10. GGUF ({best_qt})',
        'Model': f'Qwen2.5-7B-{best_qt}',
        'Primary Metric': 'BERTScore',
        'Score': qt_info.get('bertscore', gguf.get('best_bertscore', 0)),
        'Secondary': f"{qt_info.get('tokens_per_sec', 0)} tok/s, {qt_info.get('size_gb', 0)}GB",
    })

master_df = pd.DataFrame(master_rows)

if len(master_df) > 0:
    # Plotly Table
    phase_colors = {
        'Classification': '#e3f2fd',
        'Retrieval': '#e8f5e9',
        'Generation': '#fff3e0',
        'Deployment': '#fce4ec',
    }
    row_colors = [phase_colors.get(p, '#ffffff') for p in master_df['Phase']]

    fig = go.Figure(go.Table(
        header=dict(
            values=['Phase', 'Stage', 'Model', 'Metric', 'Score', 'Additional'],
            fill_color='#1565c0',
            font=dict(color='white', size=13),
            align='left',
        ),
        cells=dict(
            values=[
                master_df['Phase'], master_df['Stage'], master_df['Model'],
                master_df['Primary Metric'],
                [f'{v:.4f}' if isinstance(v, (int, float)) else str(v) for v in master_df['Score']],
                master_df['Secondary'],
            ],
            fill_color=[row_colors],
            align='left',
            font=dict(size=12),
            height=30,
        ),
    ))
    fig.update_layout(
        title='Master Table — Full Pipeline Key Metrics',
        width=1000, height=max(300, 50 * len(master_df) + 100),
        margin=dict(t=60, b=20),
    )
    fig.show(renderer='iframe')
else:
    print("No result files available to build master table.")

master_df

---
## 3. Progression Curve — Phase별 성능 개선 추이

In [ ]:
# --- Progression Curve: Bar + Line ---
if len(master_df) > 0:
    stages = master_df['Stage'].tolist()
    scores = master_df['Score'].tolist()

    phase_color_map = {
        'Classification': '#42a5f5',
        'Retrieval': '#66bb6a',
        'Generation': '#ffa726',
        'Deployment': '#ef5350',
    }
    bar_colors = [phase_color_map.get(p, '#888') for p in master_df['Phase']]

    fig = go.Figure()

    # Bar chart
    fig.add_trace(go.Bar(
        x=stages, y=scores,
        marker_color=bar_colors,
        text=[f'{v:.4f}' for v in scores],
        textposition='outside',
        name='Score',
        showlegend=False,
    ))

    # Connecting line
    fig.add_trace(go.Scatter(
        x=stages, y=scores,
        mode='lines+markers',
        line=dict(color='gray', width=1.5, dash='dot'),
        marker=dict(size=8, color='gray'),
        name='Trend',
        showlegend=False,
    ))

    # Phase annotations
    phases_seen = set()
    for i, (stage, phase) in enumerate(zip(stages, master_df['Phase'])):
        if phase not in phases_seen:
            phases_seen.add(phase)
            fig.add_annotation(
                x=stage, y=max(scores) * 1.15,
                text=f'<b>{phase}</b>',
                showarrow=False,
                font=dict(size=10, color=phase_color_map.get(phase, '#888')),
            )

    fig.update_layout(
        title='Pipeline Progression — Score by Stage',
        yaxis_title='Score',
        yaxis_range=[0, max(scores) * 1.25 if scores else 1],
        xaxis_tickangle=-25,
        width=1000, height=500,
        margin=dict(t=80, b=100),
        annotations=[
            dict(
                text='Classification: Macro F1 | Retrieval: Recall@5 | Generation/Deploy: BERTScore',
                xref='paper', yref='paper', x=0.5, y=-0.2,
                showarrow=False, font=dict(size=10, color='gray'),
            ),
        ],
    )
    fig.show(renderer='iframe')
else:
    print("No data available for progression chart.")

---
## 4. Waterfall Chart — 각 단계 개선율 (Delta %)

In [ ]:
# --- Waterfall Chart: phase-over-phase improvement ---
# Group by phase and compute deltas within each phase
waterfall_data = []

# Classification: ML → DL
if results['ml_classification'] and results['dl_classification']:
    ml_f1 = results['ml_classification']['best_models']['domain']['macro_f1']
    dl_f1 = results['dl_classification']['best_models']['domain']['macro_f1']
    waterfall_data.append({'stage': '04. ML Baseline', 'value': ml_f1, 'type': 'absolute'})
    waterfall_data.append({'stage': '05. ML→DL', 'value': dl_f1 - ml_f1, 'type': 'relative'})

# Retrieval: BM25 → SBERT
if results['bm25_retrieval'] and results['sbert_retrieval']:
    bm25_r5 = results['bm25_retrieval'].get('recall_at_5',
              results['bm25_retrieval'].get('metrics', {}).get('recall_at_5', 0))
    sbert_r5 = results['sbert_retrieval'].get('recall_at_5', 0)
    waterfall_data.append({'stage': '06. BM25', 'value': bm25_r5, 'type': 'absolute'})
    waterfall_data.append({'stage': '07. BM25→SBERT', 'value': sbert_r5 - bm25_r5, 'type': 'relative'})

# Generation: RAG → LoRA
if results['rag_generation'] and results['lora_generation']:
    rag_bs = results['rag_generation']['metrics'].get('rag_bertscore', 0)
    lora_bs = results['lora_generation']['metrics'].get('lora_bertscore', 0)
    waterfall_data.append({'stage': '08. RAG', 'value': rag_bs, 'type': 'absolute'})
    waterfall_data.append({'stage': '09. RAG→LoRA', 'value': lora_bs - rag_bs, 'type': 'relative'})

if waterfall_data:
    # Build waterfall using go.Waterfall
    fig = go.Figure(go.Waterfall(
        x=[d['stage'] for d in waterfall_data],
        y=[d['value'] for d in waterfall_data],
        measure=[d['type'] if d['type'] != 'absolute' else 'absolute' for d in waterfall_data],
        text=[f"{d['value']:+.4f}" if d['type'] == 'relative' else f"{d['value']:.4f}" for d in waterfall_data],
        textposition='outside',
        increasing=dict(marker=dict(color='#66bb6a')),
        decreasing=dict(marker=dict(color='#ef5350')),
        totals=dict(marker=dict(color='#42a5f5')),
        connector=dict(line=dict(color='gray', width=1, dash='dot')),
    ))

    fig.update_layout(
        title='Waterfall — Per-Stage Improvement Delta',
        yaxis_title='Score / Delta',
        width=900, height=500,
        xaxis_tickangle=-25,
        margin=dict(t=80, b=100),
    )
    fig.show(renderer='iframe')
else:
    print("Insufficient data for waterfall chart.")

---
## 5. Quality vs Speed — 모델별 트레이드오프

In [ ]:
# --- Quality vs Speed bubble scatter ---
bubble_data = []

# RAG generation
if results['rag_generation']:
    rag = results['rag_generation']
    rag_lat = rag.get('latency', {}).get('rag_mean_s', 5.0)
    bubble_data.append({
        'Model': f"RAG ({rag.get('llm_model', 'Ollama')})",
        'BERTScore': rag['metrics'].get('rag_bertscore', 0),
        'Speed (1/latency)': 1.0 / rag_lat if rag_lat > 0 else 0,
        'Category': 'RAG',
        'Size': 30,
    })

# GGUF models
if results['gguf_deploy']:
    for qt_name, qt_info in results['gguf_deploy'].get('quantizations', {}).items():
        lat = qt_info.get('avg_latency_s', 5.0)
        bubble_data.append({
            'Model': f'GGUF {qt_name}',
            'BERTScore': qt_info.get('bertscore', 0),
            'Speed (1/latency)': 1.0 / lat if lat > 0 else 0,
            'Category': 'GGUF',
            'Size': qt_info.get('size_gb', 4) * 8,  # bubble size proportional to model size
        })

if bubble_data:
    bdf = pd.DataFrame(bubble_data)

    color_map = {'RAG': '#ffa726', 'GGUF': '#42a5f5'}

    fig = go.Figure()
    for cat in bdf['Category'].unique():
        sub = bdf[bdf['Category'] == cat]
        fig.add_trace(go.Scatter(
            x=sub['Speed (1/latency)'],
            y=sub['BERTScore'],
            mode='markers+text',
            text=sub['Model'],
            textposition='top center',
            marker=dict(
                size=sub['Size'],
                color=color_map.get(cat, '#888'),
                opacity=0.7,
                line=dict(width=1, color='white'),
            ),
            name=cat,
        ))

    fig.update_layout(
        title='Quality vs Speed — Model Trade-off',
        xaxis_title='Speed (1/latency, higher = faster)',
        yaxis_title='BERTScore F1',
        width=850, height=500,
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    fig.show(renderer='iframe')
else:
    print("No generation/deployment results available for quality-speed analysis.")

---
## 6. Per-Domain Heatmap — 도메인별 검색 성능 편차

In [ ]:
# --- Per-Domain Retrieval Heatmap ---
heatmap_data = {}

# BM25 per-domain
if results['bm25_retrieval'] and 'per_domain' in results['bm25_retrieval']:
    bm25_pd = results['bm25_retrieval']['per_domain']
    for domain, info in bm25_pd.items():
        if domain not in heatmap_data:
            heatmap_data[domain] = {}
        metrics = info.get('metrics', info)
        # Handle both {5: {recall: ...}} and {'5': {recall: ...}} formats
        r5 = metrics.get(5, metrics.get('5', {})).get('recall', 0)
        heatmap_data[domain]['BM25 Recall@5'] = r5

# SBERT per-domain (if available in experiments)
if results['sbert_retrieval'] and 'experiments' in results['sbert_retrieval']:
    # SBERT results may not have per-domain — skip gracefully
    pass

if heatmap_data:
    domains = sorted(heatmap_data.keys())
    methods = sorted(set(m for d in heatmap_data.values() for m in d.keys()))

    z = []
    for domain in domains:
        row = [heatmap_data[domain].get(m, 0) for m in methods]
        z.append(row)

    fig = go.Figure(go.Heatmap(
        z=z, x=methods, y=domains,
        colorscale='Viridis',
        text=[[f'{v:.3f}' for v in row] for row in z],
        texttemplate='%{text}',
        textfont=dict(size=11),
    ))
    fig.update_layout(
        title='Per-Domain Retrieval Performance Heatmap',
        xaxis_title='Method', yaxis_title='Domain',
        width=600, height=max(400, 30 * len(domains)),
        margin=dict(t=60),
    )
    fig.show(renderer='iframe')
else:
    print("Per-domain data not available. Run notebooks 06/07 with domain-level evaluation.")

---
## 7. Radar Chart — ML Baseline vs DL vs Full Pipeline

In [ ]:
# --- Radar Chart: multi-model comparison ---
radar_configs = []

# Build radar data with available results
if results['ml_classification']:
    best_ml = results['ml_classification']['best_models']['domain']
    radar_configs.append({
        'name': f"ML ({best_ml['model']})",
        'color': '#42a5f5',
        'values': {
            'Domain F1': best_ml['macro_f1'],
            'Domain Acc': best_ml.get('accuracy', 0),
            'W-F1': best_ml.get('weighted_f1', 0),
        },
    })

if results['dl_classification']:
    best_dl = results['dl_classification']['best_models']['domain']
    radar_configs.append({
        'name': 'DL (KoELECTRA)',
        'color': '#66bb6a',
        'values': {
            'Domain F1': best_dl['macro_f1'],
            'Domain Acc': best_dl.get('accuracy', 0),
            'W-F1': best_dl.get('weighted_f1', 0),
        },
    })

# Add retrieval metrics if available
if results['sbert_retrieval']:
    sbert = results['sbert_retrieval']
    for cfg in radar_configs:
        cfg['values']['Retrieval R@5'] = 0  # Classification models don't have retrieval
    # Separate retrieval radar entry
    radar_configs.append({
        'name': f"SBERT ({sbert.get('best_model', '')})",
        'color': '#ffa726',
        'values': {
            'Domain F1': 0,
            'Domain Acc': 0,
            'W-F1': 0,
            'Retrieval R@5': sbert.get('recall_at_5', 0),
        },
    })

# Normalize: show classification models in a clean radar
cls_configs = [c for c in radar_configs if 'ML' in c['name'] or 'DL' in c['name']]

if len(cls_configs) >= 2:
    categories = list(cls_configs[0]['values'].keys())

    fig = go.Figure()
    for cfg in cls_configs:
        vals = [cfg['values'].get(c, 0) for c in categories]
        vals.append(vals[0])  # close polygon
        fig.add_trace(go.Scatterpolar(
            r=vals,
            theta=categories + [categories[0]],
            fill='toself',
            name=cfg['name'],
            line=dict(color=cfg['color']),
            opacity=0.6,
        ))

    fig.update_layout(
        title='Classification — ML Baseline vs Deep Learning',
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        width=700, height=550,
    )
    fig.show(renderer='iframe')
else:
    print("Need both ML and DL results for radar comparison.")

---
## 8. GGUF Pareto — 양자화별 품질-크기 분석

In [ ]:
# --- GGUF Pareto: BERTScore vs Model Size ---
if results['gguf_deploy'] and results['gguf_deploy'].get('quantizations'):
    quants = results['gguf_deploy']['quantizations']

    qt_names = list(quants.keys())
    sizes = [quants[n].get('size_gb', 0) for n in qt_names]
    bert_scores = [quants[n].get('bertscore', 0) for n in qt_names]
    speeds = [quants[n].get('tokens_per_sec', 0) for n in qt_names]

    fig = go.Figure()

    # Scatter points
    fig.add_trace(go.Scatter(
        x=sizes, y=bert_scores,
        mode='markers+text',
        text=[f'{n}\n({s:.1f} tok/s)' for n, s in zip(qt_names, speeds)],
        textposition='top center',
        marker=dict(
            size=[max(10, s * 2) for s in speeds],
            color=speeds,
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title='tok/s'),
            line=dict(width=1, color='white'),
        ),
        showlegend=False,
    ))

    # Pareto frontier line (sort by size ascending)
    sorted_idx = sorted(range(len(sizes)), key=lambda i: sizes[i])
    pareto_sizes = []
    pareto_scores = []
    best_score = -1
    for i in sorted_idx:
        if bert_scores[i] > best_score:
            best_score = bert_scores[i]
            pareto_sizes.append(sizes[i])
            pareto_scores.append(bert_scores[i])

    if len(pareto_sizes) > 1:
        fig.add_trace(go.Scatter(
            x=pareto_sizes, y=pareto_scores,
            mode='lines',
            line=dict(color='red', width=1.5, dash='dash'),
            name='Pareto Frontier',
        ))

    fig.update_layout(
        title='GGUF Quantization — Quality vs Size (Pareto)',
        xaxis_title='Model Size (GB)',
        yaxis_title='BERTScore F1',
        width=800, height=500,
    )
    fig.show(renderer='iframe')
else:
    print("GGUF results not available for Pareto analysis.")

---
## 9. Summary — 최종 결론 및 추천 배포 구성

In [ ]:
# === 전체 파이프라인 Sankey Diagram ===
# 분류 → 검색 → 생성 → 배포 흐름 + 각 단계 핵심 지표

cls_ml_score = results['ml_classification']['best_models']['domain']['macro_f1'] if results['ml_classification'] else 0
cls_dl_score = results['dl_classification']['best_models']['domain']['macro_f1'] if results['dl_classification'] else 0
ret_bm25_score = results['bm25_retrieval'].get('recall_at_5', results['bm25_retrieval'].get('metrics', {}).get('recall_at_5', 0)) if results['bm25_retrieval'] else 0
ret_sbert_score = results['sbert_retrieval'].get('recall_at_5', 0) if results['sbert_retrieval'] else 0
gen_rag_score = results['rag_generation']['metrics'].get('rag_bertscore', 0) if results['rag_generation'] else 0
gen_lora_score = results['lora_generation']['metrics'].get('lora_bertscore', 0) if results['lora_generation'] else 0
deploy_score = 0
if results['gguf_deploy']:
    best_qt = results['gguf_deploy'].get('best_quantization', '')
    deploy_score = results['gguf_deploy'].get('quantizations', {}).get(best_qt, {}).get('bertscore', 0)

labels = [
    '원본 데이터',
    f'ML Baseline\nF1={cls_ml_score:.3f}' if cls_ml_score else 'ML Baseline',
    f'KoELECTRA\nF1={cls_dl_score:.3f}' if cls_dl_score else 'KoELECTRA',
    f'BM25\nR@5={ret_bm25_score:.3f}' if ret_bm25_score else 'BM25',
    f'SBERT+FAISS\nR@5={ret_sbert_score:.3f}' if ret_sbert_score else 'SBERT+FAISS',
    f'RAG\nBS={gen_rag_score:.3f}' if gen_rag_score else 'RAG',
    f'QLoRA\nBS={gen_lora_score:.3f}' if gen_lora_score else 'QLoRA',
    f'GGUF 배포\nBS={deploy_score:.3f}' if deploy_score else 'GGUF 배포',
]

# 흐름: 원본 → 분류 → 검색 → 생성 → 배포
flow_data = [
    (0, 1, 10, '#bbdefb'),   # 원본 → ML Baseline
    (1, 2, 10, '#c8e6c9'),   # ML → DL (개선)
    (0, 3, 8, '#ffe0b2'),    # 원본 → BM25
    (3, 4, 8, '#fff9c4'),    # BM25 → SBERT (개선)
    (2, 5, 6, '#f8bbd0'),    # 분류 → RAG
    (4, 5, 6, '#e1bee7'),    # 검색 → RAG
    (5, 6, 6, '#d1c4e9'),    # RAG → LoRA (개선)
    (6, 7, 6, '#ffccbc'),    # LoRA → GGUF 배포
]

node_colors = ['#90caf9', '#42a5f5', '#1565c0', '#ffa726', '#ff9800',
               '#ef5350', '#c62828', '#ad1457']

fig = go.Figure(go.Sankey(
    node=dict(
        pad=20, thickness=25,
        label=labels,
        color=node_colors[:len(labels)],
    ),
    link=dict(
        source=[f[0] for f in flow_data],
        target=[f[1] for f in flow_data],
        value=[f[2] for f in flow_data],
        color=[f[3] for f in flow_data],
    ),
))

fig.update_layout(
    title='전체 파이프라인 흐름 및 단계별 핵심 지표',
    width=950, height=500,
    margin=dict(t=60, b=30),
)
fig.show(renderer='iframe')

In [ ]:
# === 모델 간 성능 범위 비교 테이블 ===
# 각 단계 best model의 score 및 모델 간 score 범위 출력

ci_rows = []

# 분류
if results['ml_classification']:
    ml_test = [r for r in results['ml_classification']['test'] if r['level'] == 'domain']
    ml_scores = [r['macro_f1'] for r in ml_test]
    ci_rows.append({
        'Phase': '분류', 'Stage': 'ML Baseline (최고)',
        'Metric': 'Macro F1', 'Score': max(ml_scores),
        'Min': min(ml_scores), 'Max': max(ml_scores),
        'N_Models': len(ml_scores),
    })

if results['dl_classification']:
    dl_test = [r for r in results['dl_classification']['test'] if r['level'] == 'domain']
    dl_scores = [r['macro_f1'] for r in dl_test]
    ci_rows.append({
        'Phase': '분류', 'Stage': 'DL (KoELECTRA)',
        'Metric': 'Macro F1', 'Score': max(dl_scores),
        'Min': min(dl_scores), 'Max': max(dl_scores),
        'N_Models': len(dl_scores),
    })

# 검색
if results['bm25_retrieval']:
    bm25_m = results['bm25_retrieval'].get('metrics', {})
    ci_rows.append({
        'Phase': '검색', 'Stage': 'BM25',
        'Metric': 'Recall@5', 'Score': bm25_m.get('recall_at_5', 0),
        'Min': bm25_m.get('recall_at_1', 0), 'Max': bm25_m.get('recall_at_10', 0),
        'N_Models': 1,
    })

if results['sbert_retrieval']:
    sbert = results['sbert_retrieval']
    sbert_exps = sbert.get('experiments', [])
    sbert_scores = [e.get('recall_at_5', 0) for e in sbert_exps if e.get('recall_at_5')] if sbert_exps else []
    ci_rows.append({
        'Phase': '검색', 'Stage': f"SBERT ({sbert.get('best_model', 'best')})",
        'Metric': 'Recall@5', 'Score': sbert.get('recall_at_5', 0),
        'Min': min(sbert_scores) if sbert_scores else sbert.get('recall_at_5', 0),
        'Max': max(sbert_scores) if sbert_scores else sbert.get('recall_at_5', 0),
        'N_Models': len(sbert_scores) if sbert_scores else 1,
    })

# 생성
if results['rag_generation']:
    rag_m = results['rag_generation']['metrics']
    ci_rows.append({
        'Phase': '생성', 'Stage': 'RAG 파이프라인',
        'Metric': 'BERTScore', 'Score': rag_m.get('rag_bertscore', 0),
        'Min': rag_m.get('rag_rouge_l', 0), 'Max': rag_m.get('rag_bertscore', 0),
        'N_Models': 1,
    })

if results['lora_generation']:
    lora_m = results['lora_generation']['metrics']
    ci_rows.append({
        'Phase': '생성', 'Stage': 'QLoRA 파인튜닝',
        'Metric': 'BERTScore', 'Score': lora_m.get('lora_bertscore', 0),
        'Min': lora_m.get('lora_rouge_l', 0), 'Max': lora_m.get('lora_bertscore', 0),
        'N_Models': 1,
    })

# 배포
if results['gguf_deploy'] and results['gguf_deploy'].get('quantizations'):
    quants = results['gguf_deploy']['quantizations']
    qt_scores = [v.get('bertscore', 0) for v in quants.values()]
    best_qt = results['gguf_deploy'].get('best_quantization', 'N/A')
    ci_rows.append({
        'Phase': '배포', 'Stage': f'GGUF ({best_qt})',
        'Metric': 'BERTScore',
        'Score': quants.get(best_qt, {}).get('bertscore', max(qt_scores) if qt_scores else 0),
        'Min': min(qt_scores) if qt_scores else 0,
        'Max': max(qt_scores) if qt_scores else 0,
        'N_Models': len(qt_scores),
    })

if ci_rows:
    ci_df = pd.DataFrame(ci_rows)

    phase_colors = {
        '분류': '#e3f2fd', '검색': '#e8f5e9',
        '생성': '#fff3e0', '배포': '#fce4ec',
    }
    row_bg = [phase_colors.get(p, '#fff') for p in ci_df['Phase']]

    fig = go.Figure(go.Table(
        header=dict(
            values=['단계', '모델', '지표', '최고 점수', '범위 (최소 - 최대)', '모델 수'],
            fill_color='#1565c0', font=dict(color='white', size=12), align='left',
        ),
        cells=dict(
            values=[
                ci_df['Phase'], ci_df['Stage'], ci_df['Metric'],
                [f"{s:.4f}" for s in ci_df['Score']],
                [f"{mn:.4f} - {mx:.4f}" for mn, mx in zip(ci_df['Min'], ci_df['Max'])],
                ci_df['N_Models'],
            ],
            fill_color=[row_bg], align='left', font=dict(size=11), height=28,
        ),
    ))
    fig.update_layout(
        title='단계별 최고 모델 성능 및 범위',
        width=950, height=max(250, 45 * len(ci_df) + 80),
        margin=dict(t=60, b=20),
    )
    fig.show(renderer='iframe')

    print("참고: 범위는 각 단계에서 비교한 모델 변형 또는 k값(예: Recall@1~Recall@10)의 최소-최대를 표시합니다.")
else:
    print("비교 테이블 생성에 필요한 결과 파일이 없습니다.")

In [ ]:
print("=" * 75)
print("     11. Results Summary — Final Conclusions")
print("=" * 75)

print("\n[Phase 1: Classification]")
if results['ml_classification'] and results['dl_classification']:
    ml_f1 = results['ml_classification']['best_models']['domain']['macro_f1']
    dl_f1 = results['dl_classification']['best_models']['domain']['macro_f1']
    print(f"  ML Baseline → KoELECTRA: Macro F1 {ml_f1:.4f} → {dl_f1:.4f} ({dl_f1 - ml_f1:+.4f})")
    print(f"  Recommendation: KoELECTRA for domain classification")
else:
    print("  Results not available")

print("\n[Phase 2: Retrieval]")
if results['bm25_retrieval'] and results['sbert_retrieval']:
    bm25_r5 = results['bm25_retrieval'].get('recall_at_5',
              results['bm25_retrieval'].get('metrics', {}).get('recall_at_5', 0))
    sbert_r5 = results['sbert_retrieval'].get('recall_at_5', 0)
    best_model = results['sbert_retrieval'].get('best_model', 'ko-sbert-nli')
    print(f"  BM25 → SBERT: Recall@5 {bm25_r5:.4f} → {sbert_r5:.4f} ({sbert_r5 - bm25_r5:+.4f})")
    print(f"  Recommendation: {best_model} + FAISS IndexFlatIP")
else:
    print("  Results not available")

print("\n[Phase 3: Generation]")
if results['lora_generation']:
    lora = results['lora_generation']
    print(f"  Base → QLoRA: BERTScore {lora['metrics'].get('base_bertscore', 0):.4f} → {lora['metrics'].get('lora_bertscore', 0):.4f}")
    print(f"  LoRA config: r={lora.get('lora_r', 16)}, alpha={lora.get('lora_alpha', 32)}")
else:
    print("  Results not available")

print("\n[Phase 4: Deployment]")
if results['gguf_deploy']:
    gguf = results['gguf_deploy']
    best_qt = gguf.get('best_quantization', 'N/A')
    qt_info = gguf.get('quantizations', {}).get(best_qt, {})
    print(f"  Best quality: {best_qt} (BERTScore={qt_info.get('bertscore', 0):.4f}, {qt_info.get('size_gb', 0):.1f}GB)")
    print(f"  Speed: {qt_info.get('tokens_per_sec', 0):.1f} tokens/sec")
else:
    print("  Results not available")

print("\n" + "=" * 75)
print("  Recommended Deployment Configuration:")
print("  " + "-" * 50)
print("  1. Classifier  : KoELECTRA (domain + category)")
print("  2. Retriever   : SBERT + FAISS IndexFlatIP")
print("  3. Generator   : QLoRA fine-tuned Qwen2.5-7B")
print("  4. Serving     : GGUF Q5_K_M (quality-size balance)")
print("  5. Fallback    : Gemini API (cloud, high-quality)")
print("=" * 75)

---
## 10. Lessons Learned & Future Work

### 한계점

1. **데이터 편향**: QA 코퍼스의 47%가 K쇼핑 도메인에 집중. 도메인 필터링으로 검색 단계에서 완화했으나, 소수 도메인의 QA 품질은 여전히 제한적
2. **평가 한계**: BERTScore/ROUGE-L은 표면적 유사도만 측정. 실제 민원 답변의 **정확성/적절성**은 사람 평가(human evaluation)가 필요
3. **단일 언어**: 한국어 전용 파이프라인으로 다국어 확장 미고려
4. **실시간 성능**: GGUF 양자화로 CPU 서빙은 가능하나, 동시 다수 사용자 처리 시 latency 증가 가능

### 개선 방향

| 우선순위 | 개선 항목 | 기대 효과 |
|---------|----------|----------|
| 1 | Human Evaluation 추가 | 자동 지표와 사람 판단 간 상관관계 검증 |
| 2 | Hybrid Retrieval (BM25 + SBERT RRF) 적용 | 키워드 + 의미 검색 결합으로 Recall 추가 개선 |
| 3 | Continual Learning 파이프라인 | 새로운 민원 유형 자동 반영 |
| 4 | vLLM / TGI 서빙 최적화 | 동시 처리량(throughput) 향상 |
| 5 | 소수 도메인 데이터 증강 | Paraphrase generation으로 소수 클래스 보강 |

### 면접 포인트

- **"왜 KoELECTRA?"** → ELECTRA의 RTD가 토큰당 학습 신호가 풍부하여 짧은 민원 텍스트에 효과적
- **"BM25로 충분하지 않나?"** → 동의어/유의어 문제(어휘 불일치)가 있어 의미 검색이 필수
- **"GGUF 양자화 손실은?"** → Q5_K_M 기준 BERTScore 저하 최소, 모델 크기 60%+ 감소
- **"다음에 할 일은?"** → Human evaluation + Hybrid retrieval + 소수 도메인 데이터 증강

---

**End of Pipeline Summary**

| Notebook | Phase | Key Result |
|----------|-------|------------|
| 01 | Data Exploration | 352K samples, 14 domains, 63 categories |
| 02 | Preprocessing | Train/Val/Test split, QA pair extraction |
| 03 | Task Definition | Multi-level classification + RAG task design |
| 04 | ML Baseline | TF-IDF + LR/SVC/LGBM baseline |
| 05 | Deep Classification | KoELECTRA > KoBERT > ML baseline |
| 06 | BM25 Retrieval | Keyword-based retrieval baseline |
| 07 | SBERT + FAISS | Semantic search improves over BM25 |
| 08 | RAG Pipeline | RAG > Zero-shot generation |
| 09 | QLoRA Fine-tuning | Domain-adapted LLM improves quality |
| 10 | GGUF Deploy | Quantized model for efficient serving |
| **11** | **Results Summary** | **This notebook — full pipeline overview** |